# 🧠 AI Trinh Sát: Deep Learning Bi-LSTM (Ultimate Local Data Fusion)

**Mục tiêu:** Quét toàn bộ dữ liệu từ ổ cứng Local (Ổ D:), dung hợp các nguồn Kaggle (XSS, Command Injection, JSON Web Payloads) và HttpParamsDataset để tạo ra lõi WAF .

## 📦 1. Import Thư viện

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import json
import re
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print(" Deep Learning Bi-LSTM (Ultimate Local Data Fusion)...")

: 

## 📂 2. dữ liệu từ Ổ D:

In [2]:
print("📂 Đang tiến hành hút và dung hợp dữ liệu...")
dfs = []

# --- A. Xử lý nhóm file HttpParamsDataset ---
def normalize_http_label(label):
    label = str(label).lower().strip()
    if label == 'norm': return 'Normal'
    if 'xss' in label or 'js-syntax' in label: return 'XSS'  # Gộp JS-Syntax vào XSS
    if 'sql' in label: return 'SQLi'
    if 'cmd' in label or 'exec' in label: return 'Command Injection'
    if 'path' in label or 'traversal' in label: return 'Path Traversal'
    if 'ssrf' in label: return 'SSRF'
    return label.upper()

http_files = [
    r"D:\AI\clawweb\data\httpparagram\payload_train.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test_lexical.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_full.csv"
]

for f in http_files:
    if os.path.exists(f):
        df_temp = pd.read_csv(f)
        df_temp = df_temp[['payload', 'attack_type']].rename(columns={'payload': 'text', 'attack_type': 'label'})
        df_temp['label'] = df_temp['label'].apply(normalize_http_label)
        dfs.append(df_temp)
        print(f"✅ Đã nạp {len(df_temp)} dòng từ {os.path.basename(f)}")
    else:
        print(f"⚠️ Không tìm thấy: {f}")

# --- B. Xử lý bộ XSS Kaggle ---
path_xss = r"D:\AI\clawweb\data\xss\XSS_dataset.csv"
if os.path.exists(path_xss):
    df_xss = pd.read_csv(path_xss)
    df_xss = df_xss[['Sentence', 'Label']].rename(columns={'Sentence': 'text', 'Label': 'label'})
    df_xss['label'] = df_xss['label'].map({1: 'XSS', 0: 'Normal'})
    dfs.append(df_xss)
    print(f"✅ Đã nạp {len(df_xss)} dòng từ XSS_dataset.csv")

# --- C. Xử lý bộ Command Injection Kaggle ---
path_cmd = r"D:\AI\clawweb\data\oscommand\command injection.csv"
if os.path.exists(path_cmd):
    df_cmd = pd.read_csv(path_cmd)
    df_cmd = df_cmd[['sentence', 'Label']].rename(columns={'sentence': 'text', 'Label': 'label'})
    df_cmd['label'] = df_cmd['label'].map({1: 'Command Injection', 0: 'Normal'})
    dfs.append(df_cmd)
    print(f"✅ Đã nạp {len(df_cmd)} dòng từ command injection.csv")

# --- D. Xử lý bộ SQL Injection (Dự đoán đường dẫn) ---
path_sqli = r"D:\AI\clawweb\data\sqli\Modified_SQL_Dataset.csv"
if not os.path.exists(path_sqli):
    path_sqli = r"D:\AI\clawweb\data\sql_injection\Modified_SQL_Dataset.csv"

if os.path.exists(path_sqli):
    df_sqli = pd.read_csv(path_sqli)
    df_sqli = df_sqli.rename(columns={"Query": "text", "Label": "label"})[["text", "label"]]
    df_sqli["label"] = df_sqli["label"].map({1: "SQLi", 0: "Normal", "1": "SQLi", "0": "Normal"})
    dfs.append(df_sqli)
    print(f"✅ Đã nạp {len(df_sqli)} dòng từ Modified_SQL_Dataset.csv")

# --- E. Xử lý JSONL (Dùng Regex Bất Tử chống lỗi file Kaggle) ---
path_jsonl = r"D:\AI\clawweb\data\web_payloads\WEB_APPLICATION_PAYLOADS.jsonl"
if os.path.exists(path_jsonl):
    json_payloads = []
    with open(path_jsonl, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Dùng Biểu thức chính quy (Regex) để cào dữ liệu thay vì json.loads
    # Việc này giúp bỏ qua hoàn toàn các dòng bị lỗi thiếu dấu phẩy của Kaggle
    matches = re.finditer(r'"payload"\s*:\s*"(.*?)"[\s\S]*?"type"\s*:\s*"(.*?)"', content)
    for match in matches:
        payload_text = match.group(1).replace('\\"', '"')
        payload_type = match.group(2).upper()
        if 'SSRF' in payload_type:
            json_payloads.append((payload_text, 'SSRF'))
        elif 'CSRF' in payload_type:
            json_payloads.append((payload_text, 'CSRF'))
    if json_payloads:
        df_json = pd.DataFrame(json_payloads, columns=["text", "label"])
        df_json = pd.concat([df_json] * 50, ignore_index=True) # Nhân bản lên 50 lần
        dfs.append(df_json)
        print(f"✅ Đã cào & nhân bản {len(df_json)} dòng SSRF/CSRF bằng Regex Engine")
    else:
        print("⚠️ Không trích xuất được SSRF/CSRF nào từ Regex.")

# --- GOM TẤT CẢ VÀ DỌN DẸP ---
if not dfs:
    raise ValueError("❌ LỖI NGHIÊM TRỌNG: Không đọc được bất kỳ file nào từ ổ D:!")

df = pd.concat(dfs, ignore_index=True)
df = df.drop_duplicates().dropna() # Xoa sach rac trung lap
df['text'] = df['text'].astype(str)

initial_len = len(df)

print(f"\n🧹 Đã dọn dẹp {initial_len - len(df)} dòng copy/paste trùng lặp.")
print(f"📊 Tổng lực lượng Data hiện có: {len(df)} dòng.")
print("\n🔥 Phân bổ Các Nhãn Lỗ Hổng (Đã Sạch Sẽ):")
print(df['label'].value_counts())

# Bơm thêm URL sạch vào lớp Normal để AI không nhận nhầm URL thành XSS/SSRF
normal_urls = pd.DataFrame({
    'text': [
        'https://www.google.com/search?q=cat',
        'http://localhost:5173/dashboard/users?id=123',
        'https://uet.vnu.edu.vn/category/tin-tuc/',
        'http://portal.edu.vn/api/docs/file.pdf',
        'https://github.com/search?q=machine+learning'
    ] * 10, # CHỈ NHÂN BẢN 10 LẦN
    'label': ['Normal'] * 50
})

df = pd.concat([df, normal_urls], ignore_index=True)


📂 Đang tiến hành hút và dung hợp dữ liệu...
✅ Đã nạp 20712 dòng từ payload_train.csv
✅ Đã nạp 10355 dòng từ payload_test.csv
✅ Đã nạp 1106 dòng từ payload_test_lexical.csv
✅ Đã nạp 31067 dòng từ payload_full.csv
✅ Đã nạp 13686 dòng từ XSS_dataset.csv
✅ Đã nạp 2106 dòng từ command injection.csv
✅ Đã nạp 30919 dòng từ Modified_SQL_Dataset.csv
✅ Đã cào & nhân bản 9950 dòng SSRF/CSRF bằng Regex Engine

🧹 Đã dọn dẹp 0 dòng copy/paste trùng lặp.
📊 Tổng lực lượng Data hiện có: 67148 dòng.

🔥 Phân bổ Các Nhãn Lỗ Hổng (Đã Sạch Sẽ):
label
Normal               36359
SQLi                 21929
XSS                   7873
Command Injection      520
Path Traversal         290
CSRF                    92
SSRF                    85
Name: count, dtype: int64


## ⚖️ 3. Xử lý Lệch Dữ Liệu (Undersampling)

In [3]:
DATA_PATH = r"D:\AI\clawweb\data"
MODEL_SAVE_PATH = r"D:\AI\clawweb\model"
MAX_LEN = 150
MAX_WORDS = 10000

def load_and_balance_data():
    print("📂 Đang tải và cân bằng dữ liệu...")
         
    # ... (Code nạp dữ liệu từ D:\AI\clawweb\data) ...
    
    # Giả sử sau khi nạp ta có biến 'df'
    # df = pd.read_csv(...) 

    # --- CHIẾN THUẬT CÂN BẰNG MỚI ---
    # Thay vì nhân bản cứng 50 lần, ta nhân bản theo tỷ lệ để các lớp đạt tối thiểu 2000 mẫu
print("⚖️ Đang thực hiện cân bằng dữ liệu nâng cao...")

target_count = 3000
new_dfs = []

for label in df['label'].unique():
    temp_df = df[df['label'] == label]
    current_count = len(temp_df)
    
    if label == 'Normal':
        # Cắt bớt Normal
        temp_df = temp_df.sample(n=min(current_count, 15000), random_state=42)
    elif current_count < target_count:
        # Nhân bản lớp thiểu số
        multiplier = int(target_count / current_count) + 1
        temp_df = pd.concat([temp_df] * multiplier, ignore_index=True).iloc[:target_count]
        # Thêm nhiễu để tránh bị xóa trùng lặp
        temp_df['text'] = temp_df['text'] + " "
    
    new_dfs.append(temp_df)

# Ghi đè trực tiếp lên biến df
df = pd.concat(new_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print("📊 Phân bổ nhãn MỚI NHẤT sau cân bằng:")
print(df['label'].value_counts())

# ===== 2. Huấn luyện hệ thống =====
def train_refined_model(df):
    le = LabelEncoder()
    y = le.fit_transform(df['label'])
    
    tokenizer = Tokenizer(num_words=MAX_WORDS, char_level=True, oov_token='<OOV>')
    tokenizer.fit_on_texts(df['text'])
    
    X = pad_sequences(tokenizer.texts_to_sequences(df['text']), maxlen=MAX_LEN)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Tính class_weight nhẹ nhàng hơn sau khi đã oversampling
    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights_dict = dict(enumerate(weights))

    # Kiến trúc Model cải tiến (thêm BatchNormalization để ổn định gradient)
    model = Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
        Bidirectional(LSTM(64, return_sequences=True)),
        BatchNormalization(),
        GlobalMaxPooling1D(),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    # Thêm EarlyStopping để tránh Overfitting vào các nhãn bị nhân bản
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    print("🚀 Bắt đầu training...")
    model.fit(
        X_train, y_train,
        epochs=15, 
        batch_size=64,
        validation_data=(X_test, y_test),
        class_weight=weights_dict,
        callbacks=[early_stop]
    )

    # Lưu trữ
    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    model.save(os.path.join(MODEL_SAVE_PATH, 'deep_learning_agent_core.keras'))
    with open(os.path.join(MODEL_SAVE_PATH, 'tokenizer.pkl'), 'wb') as f:
        pickle.dump(tokenizer, f)
    with open(os.path.join(MODEL_SAVE_PATH, 'label_encoder.pkl'), 'wb') as f:
        pickle.dump(le, f)
    print("✅ Đã cập nhật 'não' mới thành công!")


⚖️ Đang thực hiện cân bằng dữ liệu nâng cao...
📊 Phân bổ nhãn MỚI NHẤT sau cân bằng:
label
SQLi                 21929
Normal               15000
XSS                   7873
Path Traversal        3000
SSRF                  3000
Command Injection     3000
CSRF                  3000
Name: count, dtype: int64


## 🧠 4. Tiền xử lý & Chia tập (Tokenization)

In [4]:
le = LabelEncoder()
y = le.fit_transform(df['label'])
num_classes = len(le.classes_)

print("🧠 Đang xây dựng từ điển nhúng ký tự (Tokenizer)...")
MAX_WORDS = 10000  
MAX_LEN = 150      

tokenizer = Tokenizer(num_words=MAX_WORDS, char_level=True, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])

X_pad = pad_sequences(tokenizer.texts_to_sequences(df['text']), maxlen=MAX_LEN, padding='post', truncating='post')

X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42)
print(f"Tập train: {X_train.shape}, Tập test: {X_test.shape}")

🧠 Đang xây dựng từ điển nhúng ký tự (Tokenizer)...
Tập train: (45441, 150), Tập test: (11361, 150)


## 🏗️ 5. Xây dựng Mạng Nơ-ron (Bi-LSTM)

In [5]:
print("🏗️ Đang lắp ráp bộ não nơ-ron Bi-LSTM...")
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5), 
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

🏗️ Đang lắp ráp bộ não nơ-ron Bi-LSTM...


c:\Users\truon\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 🔥 6. Huấn luyện (Training)

In [6]:
from sklearn.utils import class_weight
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Tính trọng số gốc
raw_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

# ÁP DỤNG SMOOTHING (Làm trơn bằng Căn bậc 2)
smoothed_weights = np.sqrt(raw_weights)
class_weights_dict = dict(enumerate(smoothed_weights))

print("⚖️ Trọng số sau khi làm trơn:", class_weights_dict)

print("🎬 Bắt đầu huấn luyện bộ não mới (Smoothed Class Weights)...")
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, y_test),
    class_weight=class_weights_dict, # ĐƯA TRỌNG SỐ LÀM TRƠN VÀO ĐÂY
    callbacks=[early_stop]
)


⚖️ Trọng số sau khi làm trơn: {0: np.float64(1.6487612004325207), 1: np.float64(1.647382060751253), 2: np.float64(0.734860022060831), 3: np.float64(1.6341143382057666), 4: np.float64(0.6093850333775901), 5: np.float64(1.6405379012822927), 6: np.float64(1.0138036708000775)}
🎬 Bắt đầu huấn luyện bộ não mới (Smoothed Class Weights)...
Epoch 1/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 51s 68ms/step - accuracy: 0.8929 - loss: 0.3664 - val_accuracy: 0.9651 - val_loss: 0.1202
Epoch 2/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 56s 79ms/step - accuracy: 0.9808 - loss: 0.0704 - val_accuracy: 0.9884 - val_loss: 0.0397
Epoch 3/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 61s 86ms/step - accuracy: 0.9879 - loss: 0.0404 - val_accuracy: 0.9895 - val_loss: 0.0343
Epoch 4/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 56s 79ms/step - accuracy: 0.9919 - loss: 0.0276 - val_accuracy: 0.9941 - val_loss: 0.0227
Epoch 5/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 60s 84ms/step - accuracy: 0.9927 - loss: 0.0245 - val_accuracy: 0.9938 - val_loss: 0.0221
Epoch 6/20
711/7

## 💾 7. Lưu Mô hình (Xuất ra Ổ D:)

In [9]:
print("\n💾 Đang xuất file não...")
save_dir = r"D:\AI\clawweb\model"
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, 'deep_learning_agent_core.keras')) 

with open(os.path.join(save_dir, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(save_dir, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

print(f"✅ HOÀN TẤT! Sẵn sàng mang não này đi đục Web. Files đã được lưu tại: {save_dir}")


💾 Đang xuất file não...
✅ HOÀN TẤT! Sẵn sàng mang não này đi đục Web. Files đã được lưu tại: D:\AI\clawweb\model


## 🧪 8. Chạy Thử Nghiệm Thực Tế

In [11]:
def test_ai(payload):
    seq = tokenizer.texts_to_sequences([str(payload)])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred_probs = model.predict(pad, verbose=0)[0]
    pred_class = np.argmax(pred_probs)
    label = le.inverse_transform([pred_class])[0]
    confidence = pred_probs[pred_class] * 100
    print(f"Payload: {payload}\n=> Phát hiện: [ {label} ] (Độ tự tin: {confidence:.2f}%)\n")

print("--- 🛡️ TEST HACKER ---")
test_ai("admin' OR 1")
test_ai("<img src='x' onerror='alert(1)'>")
test_ai("test && cat /etc/passwd")
test_ai("../../../../etc/shadow")
test_ai("http://169.254.169.254/latest/meta-data/")

print("--- 🟢 TEST NORMAL ---")
test_ai("Xin chào, tôi muốn tìm tài liệu NCKH")
test_ai("https://www.google.com/search?q=cat")

--- 🛡️ TEST HACKER ---
Payload: admin' OR 1
=> Phát hiện: [ SQLi ] (Độ tự tin: 91.78%)

Payload: <img src='x' onerror='alert(1)'>
=> Phát hiện: [ XSS ] (Độ tự tin: 97.04%)

Payload: test && cat /etc/passwd
=> Phát hiện: [ Normal ] (Độ tự tin: 89.46%)

Payload: ../../../../etc/shadow
=> Phát hiện: [ Path Traversal ] (Độ tự tin: 97.39%)

Payload: http://169.254.169.254/latest/meta-data/
=> Phát hiện: [ Normal ] (Độ tự tin: 99.67%)

--- 🟢 TEST NORMAL ---
Payload: Xin chào, tôi muốn tìm tài liệu NCKH
=> Phát hiện: [ Normal ] (Độ tự tin: 99.55%)

Payload: https://www.google.com/search?q=cat
=> Phát hiện: [ Normal ] (Độ tự tin: 82.08%)

